In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import ndcg_score

In [ ]:
# TCVファイルの読み込み
train_A = pd.read_csv('../000.data/train/train_A.tsv', sep="\t")
train_B = pd.read_csv('../000.data/train/train_B.tsv', sep="\t")
train_C = pd.read_csv('../000.data/train/train_C.tsv', sep="\t")
train_D = pd.read_csv('../000.data/train/train_D.tsv', sep="\t")
test_data = pd.read_csv('../000.data/test.tsv', sep="\t")

In [ ]:
# trainを一つに
train_data = pd.concat([train_A, train_B, train_C, train_D], ignore_index=True)

In [ ]:
# 関連度スコアの設定
def compute_relevance(row):
    if row["event_type"] == 3 and row["ad"] == 1:  # 広告経由の購入
        return 3
    elif row["event_type"] == 2:  # 広告クリック
        return 2
    elif row["event_type"] == 1:  # 詳細ページ閲覧
        return 1
    else:  # カート追加
        return 0

train_data["relevance"] = train_data.apply(compute_relevance, axis=1)

In [ ]:
# タイムスタンプを数値型に変換
train_data["time_stamp"] = pd.to_datetime(train_data["time_stamp"])
train_data['time_stamp'] = train_data['time_stamp'].astype(np.int64) // 10**9  # UNIX時間に変換

In [ ]:
# ユーザIDと商品IDのエンコード
user_enc = LabelEncoder()
product_enc = LabelEncoder()

train_data['user_id'] = user_enc.fit_transform(train_data['user_id'])
train_data['product_id'] = product_enc.fit_transform(train_data['product_id'])

test_data[('user_id')] = user_enc.transform(test_data['user_id'])

In [ ]:
# XGBoost 用データ作成
train_X = train_data[['user_id', 'product_id', 'ad', 'time_stamp']]
train_y = train_data['relevance']

In [ ]:
# モデル学習
model = xgb.XGBRegressor(
    objective='reg:squarederror',
    learning_rate = ,
    max_depth = ,
    n_estimators = ,
    subsample = ,
    colsample_bytree = ,
    min_child_weight = ,
    gamma = ,
    reg_alpha = ,
    reg_lambda = ,
    tree_method = ,
    grow_policy = 
)
model.fit(train_X, train_y)

In [ ]:
# 推薦関数
def recommend_products(user_id, top_k=22):
    user_data = train_data[train_data['user_id'] == user_id][['user_id', 'product_id', 'ad', 'time_stamp']]
    if user_data.empty:
        print(f"Warning: User {user_id} has no data.")
        return pd.DataFrame(columns=['product_id', 'rank'])
    predictions = model.predict(user_data)
    user_data['score'] = predictions
    recommendations = user_data.sort_values(by='score', ascending=False).head(top_k)
    recommendations['rank'] = range(1, len(recommendations) + 1)
    recommendations['user_id'] = user_enc.inverse_transform(recommendations['user_id'])
    recommendations['product_id'] = product_enc.inverse_transform(recommendations['product_id'])
    return recommendations[['user_id', 'product_id', 'rank']]

In [ ]:
# nGCDの計算
def evaluate_ndcg():
    y_true = []
    y_score = []
    for user_id in test_data['user_id'].unique():
        if user_id in train_data['user_id'].values:
            actual = train_data[train_data['user_id'] == user_id].sort_values(by='relevance', ascending=False)['relevance'].values
            pred = recommend_products(user_id, top_k=len(actual))['rank'].values
            if len(pred) == 0:
                print(f"Warning: No predictions for user {user_id}.")
                continue
            y_true.append(actual)
            y_score.append(pred)
    return np.mean([ndcg_score([t], [s]) for t, s in zip(y_true, y_score)]) if y_true else 0.0

In [ ]:
# 予測結果保存
def save_predictions():
    results = []
    for user_id in test_data['user_id'].unique():
        if user_id in train_data['user_id'].values:
            recommendations = recommend_products(user_id)
            for _, row in recommendations.iterrows():
                results.append([row['user_id'], row['product_id'], row['rank']])
    if not results:
        print("Warning: No predictions generated.")
    pd.DataFrame(results, columns=['user_id', 'product_id', 'rank']).to_csv('predictions.tsv', sep='\t', index=False, header=False)

In [ ]:
# 推薦結果表示
print("nDCG Score:", evaluate_ndcg())
save_predictions()